<a href="https://colab.research.google.com/github/spielraump7/FF14_logs/blob/main/GetNewQuestList.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

最新パッチテキストのクエストリスト取得
FF14_logs フォルダに スプレッドシート FFXIV_questList_TMP として出力

# 定数定義

In [ ]:
# 定数定義

import requests
from bs4 import BeautifulSoup
import math
import re

# LodeStone DB 最新パッチクエストリストURL
URL_BASE = 'https://jp.finalfantasyxiv.com/lodestone/playguide/db/quest/?patch=latest'

# カテゴリー階層2を指定  0:メインクエスト(過去分)/1:メインクエスト(最新)/2:クロニクルクエスト
# 3:サブクエスト/4:友好部族クエスト(過去分)/5:友好部族クエスト(最新)/6:クラス・ジョブクエスト/7:その他
#CATE2 = '7'

# カテゴリー階層3を指定  ※事前にロドストで確認
#CATE3_LIST = [97]

# Unicode私的領域(レベルアジャスト記号などの特殊文字)の範囲を指定
#pua_pattern = re.compile(r'[\uE000-\uF8FF\U000F0000-\U000FFFFD\U00100000-\U0010FFFD]')
pua_pattern = re.compile(r'[\uE000-\uF8FF]')

# 関数

## getQuestName(BeautifulSoup)
### 取得したHTMLを解析して、出力用のリストに格納する関数

In [ ]:
# 取得したHTMLを解析して、出力用のリストに格納する関数
def getQuestName(soup):

  # <td>要素の該当属性を持つ箇所から、カテゴリ階層/クエスト名/IDを取得
  for e in soup.find_all("td",class_=re.compile("db-table__quest__title")):

    # カテゴリ情報記載部分を分析
    ctg_block = e.find("span", class_="db-table__txt--type")

    # ブロック内の<a>要素を全て取得し、カテゴリ2～4の名称を取得
    a_tags = ctg_block.find_all('a')
    ctg2 = a_tags[0].text.strip() if len(a_tags) > 0 else None
    ctg3 = a_tags[1].text.strip() if len(a_tags) > 1 else None
    ctg4 = a_tags[2].text.strip() if len(a_tags) > 2 else None

    # クエスト名の取得処理
    # href属性からリンクを取得、余計な部分を削除してIDだけ残す
    title_link = e.find("a", class_=re.compile("db-table__txt--detail_link"))
    lurl_list = title_link.attrs["href"].split('/')
    qid = lurl_list[len(lurl_list) -2]
    ttl = pua_pattern.sub('', title_link.get_text()).strip()

    # 兄弟要素の<td>を取得
    for sbl in e.next_siblings:
      if sbl and sbl.name == "td":

        # classに quest__area の記述があれば、受注エリア
        if "db-table__quest__area" in sbl["class"]:
          ac_area = sbl.string
        # そうでなければ受注レベル
        else:
          ac_lv = sbl.string

    # 結果出力用のリストに取得したデータを格納
    out_list.append([ctg2, ctg3, ctg4, qid, ttl, ac_area, ac_lv])

# メインプロシージャ

In [ ]:
# LodeStoneをスクレイピングして、指定のカテゴリのクエストリストを取得
# 出力用のリストに格納する

# 結果出力用の２次元配列を宣言、ヘッダを格納
out_list = [[]]
out_list[0] = (['カテゴリ2', 'カテゴリ3', 'カテゴリ4', 'ID', 'クエスト名', '受注エリア', '受注レベル'])

# カテゴリー階層3リストに指定された要素数分ループ
#for ctg3 in CATE3_LIST:

#  url = URL_BASE + '?category2=' + CATE2 + '&category3=' + str(ctg3)

# HTMLを取得してパース
soup = BeautifulSoup(requests.get(URL_BASE).content, "html.parser")

# 先頭ページに表示される分を解析
getQuestName(soup)

# 総クエスト件数を取得
for e in soup.find_all("span",class_="total"):
    total_lists = int(e.text.strip())
    break

# 総件数が50よりも大きければ、ページ数分ループする
if total_lists > 50:

  for p in range(2, math.ceil(total_lists / 50) + 1):
    url = URL_BASE + '&page=' + str(p)
    soup_p = BeautifulSoup(requests.get(url).content, "html.parser")
    getQuestName(soup_p)

# debug: 取得した要素
#print(len(out_list))

In [ ]:
# リストに格納された２次元配列をGoogleスプレッドシートに出力する

# 認証
!pip install --upgrade -q gspread google-auth

from google.colab import auth
auth.authenticate_user()

import gspread
import google.auth

credentials, project = google.auth.default()
gc = gspread.authorize(credentials)

# 出力するフォルダを指定
#FOLDER_ID = '1mtpFi2SSV-L4R0gDqaBVl4MjEBCBjKGB'
FOLDER_ID = '15f_pUUwL7gCFLnkd6YDS2-Pcam_9_uAB'

# 出力用ファイル名を指定
WB_NAME = "FFXIV_questList_TMP"

# 新しいブックを開くき、出力するシート名を変更して行数を増やしておく
sh = gc.create(WB_NAME, folder_id=FOLDER_ID)
worksheet = gc.open(WB_NAME).sheet1
worksheet.add_rows(len(out_list))

# シートの出力範囲のセルを指定して、リストの中身を出力
cell_list = worksheet.range(1, 1, len(out_list), len(out_list[0]))
col_list = [flatten for inner in out_list for flatten in inner]

for cell, col in zip(cell_list, col_list):
  cell.value = col

worksheet.update_cells(cell_list)

{'spreadsheetId': '10WTuSMeo2N8QSjFFwyATt7gvculkpI9QfzFXlQwjgME',
 'updatedRange': "'シート1'!A1:G63",
 'updatedRows': 63,
 'updatedColumns': 7,
 'updatedCells': 427}